1. Load image_manifest.csv and select images where split == "validation".
2. Use the finalized prompt and JSON Schema.
3. Save each Gemini prediction individually, with API error handling and automatic skipping of previously processed images.
4. Perform the same quantitative evaluations on all 61 validation images.
5. Generate final comparison tables for follicle-like region counts, sizes, and spatial agreement.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import json

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

manifest = pd.read_csv(MANIFEST_PATH)

assert len(manifest) == 301
assert manifest["case_id"].is_unique

print("Total dataset images:", len(manifest))

Mount Google Drive and load the dataset manifest for the main evaluation

In [ ]:
validation = (
    manifest[manifest["split"] == "validation"]
    .sort_values("case_id")
    .reset_index(drop=True)
)

assert len(validation) == 61
assert validation["case_id"].is_unique

# Make sure no pilot images are included
assert not validation["is_pilot"].astype(str).str.lower().eq("true").any()

# Verify that all images and pseudo-labels exist
for _, row in validation.iterrows():
    assert Path(row["image_path"]).exists()
    assert Path(row["pseudo_label_path"]).exists()

print("Validation images:", len(validation))

print("\nCategory distribution:")
print(validation["category"].value_counts().sort_index())

display(validation[["case_id", "category", "split"]].head(10))

Select the 61 validation cases from the original nnU-Net fold 0

In [ ]:
# Load the configuration saved during the pilot experiment.

CONFIG_PATH = (
    PROJECT_DIR / "config" / "gemini_pilot_config.json"
)

assert CONFIG_PATH.exists(), (
    f"Configuration not found: {CONFIG_PATH}"
)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

MODEL_ID = config["model_id"]
ANNOTATION_PROMPT = config["prompt"]
RESPONSE_SCHEMA = config["response_schema"]
TEMPERATURE = config["temperature"]

assert config["prompt_version"] == "v1"
assert len(config["pilot_case_ids"]) == 10
assert config["bbox_coordinate_range"] == [0, 1000]
assert TEMPERATURE == 0

print("Model:", MODEL_ID)
print("Prompt version:", config["prompt_version"])
print("Temperature:", TEMPERATURE)
print("Schema loaded:", bool(RESPONSE_SCHEMA))
print("Prompt length:", len(ANNOTATION_PROMPT))

Load and verify the frozen Gemini pilot configuration

In [ ]:
from google import genai
from google.colab import userdata
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Literal

# Use the same Pydantic schema as in the pilot experiment.
class FollicleRegion(BaseModel):
    bbox: List[int] = Field(
        description="Bounding box [xmin, ymin, xmax, ymax], normalized to 0-1000."
    )
    shape: Literal["round", "oval", "irregular", "uncertain"]
    appearance: str

class UltrasoundAnnotation(BaseModel):
    follicle_like_count: int
    regions: List[FollicleRegion]
    image_quality: Literal["good", "moderate", "poor"]
    uncertain: bool
    description: str

# Verify consistency with the saved pilot schema.
assert UltrasoundAnnotation.model_json_schema() == RESPONSE_SCHEMA, (
    "The current annotation schema differs from the pilot schema."
)

API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("GEMINI_API_KEY is missing.")

client = genai.Client(api_key=API_KEY)

# Separate directory for the main experiment.
MAIN_PREDICTIONS_DIR = (
    PROJECT_DIR / "predictions" / "main_validation"
)
MAIN_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print("Gemini client initialized.")
print("Selected model:", MODEL_ID)
print("Response schema verified.")
print("Output directory:", MAIN_PREDICTIONS_DIR)

Initialize the Gemini client and validate the saved response schema